In [1]:
import os
import torch
from optuna.integration import PyTorchLightningPruningCallback
import pandas as pd
import optuna
import numpy as np
import amplfi

device = torch.device('cuda') if torch.cuda.is_available() else 'cpu'
os.environ['AMPLFI_OUTDIR'] = '/projects/bcse/jredepenning/amplfi-outdir/run5/sg_sky_loc'
os.environ['AMPLFI_DATADIR'] = '/projects/bcse/deep1018/amplfi-data-dir-03'

/projects/bcse/jredepenning/amplfi/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
ckpt_path = "/projects/bcse/jredepenning/amplfi-outdir/run5/sg_sky_loc/train_logs/best.ckpt"
config_path = "/projects/bcse/jredepenning/amplfi/runs/sg_sky_loc/sg.yaml"

In [3]:
from amplfi.train.data.datasets.base import AmplfiDataset
from amplfi.train.models.base import AmplfiModel
from amplfi.train.cli.flow import AmplfiFlowCLI

cli = AmplfiFlowCLI(
        AmplfiModel,
        AmplfiDataset,
        subclass_mode_model=True,
        subclass_mode_data=True,
        save_config_kwargs={"overwrite": True},
        seed_everything_default=101588,
        args=[
        "test",             # Subcommand
        "--config", config_path,
        "--ckpt_path", ckpt_path,
        ]
)

/projects/bcse/jredepenning/amplfi/.venv/lib/python3.12/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS
/projects/bcse/jredepenning/amplfi/.venv/lib/python3.12/site-packages/lightning/pytorch/cli.py:528: LightningCLI's args parameter is intended to run from within Python like if it were from the command line. To prevent mistakes it is not recommended to provide both args and command line arguments, got: sys.

2025-10-03 12:11:50,841 - root - INFO - Downloading data to /projects/bcse/deep1018/amplfi-data-dir-03


ValueError: Does not validate against any of the Union subtypes
Subtypes: [<class 'NoneType'>, <class 'amplfi.train.data.datasets.base.AmplfiDataset'>]
Errors:
  - Expected a <class 'NoneType'>
  - [Errno 13] Permission denied: '/projects/bcse/deep1018/amplfi-data-dir-03'
Given value type: <class 'jsonargparse._namespace.Namespace'>
Given value: Namespace(class_path='amplfi.train.data.datasets.FlowDataset', init_args=Namespace(data_dir='/projects/bcse/deep1018/amplfi-data-dir-03', inference_params=['phase', 'dec', 'psi', 'phi'], highpass=25.0, sample_rate=2048.0, kernel_length=5.0, fduration=2.0, psd_length=12.0, batches_per_epoch=10, batch_size=128, ifos=['H1', 'L1'], waveform_sampler=Namespace(class_path='amplfi.train.data.waveforms.generator.sg.SGGenerator', init_args=Namespace(num_val_waveforms=10000, num_test_waveforms=1, parameter_sampler=Namespace(class_path='amplfi.train.data.utils.utils.ParameterSampler', init_args=Namespace(parameters={'frequency': Namespace(class_path='torch

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

def prepare_data(datamodule):
    datamodule.transforms_to_device()
    train_dataloader = datamodule.train_dataloader()
    val_dataloader = datamodule.val_dataloader()
    train_batch = []
    val_batch = []
    datamodule.trainer.training = True
    for batch in tqdm(train_dataloader):
        t_batch = datamodule.on_after_batch_transfer(batch, device)
        train_batch.append(t_batch)
    
    datamodule.trainer.training = False
    datamodule.trainer.validating = True
    for batch in tqdm(val_dataloader):
        v_batch = datamodule.on_after_batch_transfer(batch, device)
        val_batch.append(v_batch)

    train_dataloader = DataLoader(train_batch, batch_size=None)
    val_dataloader = DataLoader(val_batch, batch_size=None)

    return train_dataloader, val_dataloader

In [ ]:
def objective(trial):
    # batch_size = trial.suggest_int("batch_size", 128, 1024, log=True)
    # learning_rate = trial.suggest_float("learning_rate", 7.14e-5, 7.14e-2, log=True)
    # weight_decay = trial.suggest_float("weight_decay", 4.2e3, 4.2e5, log=True)
    batches_per_epoch = trial.suggest_int("batches_per_epoch", 2, 5)
    model = cli.model
    model.eval();
    datamodule = cli.datamodule
    datamodule.transforms_to_device()
    datamodule.setup("fit")
    model.hparams["learning_rate"] = 7.14e-1
    datamodule.hparams["batch_size"] = 64
    datamodule.hparams["batches_per_epoch"] = batches_per_epoch

    train_dataloader, val_dataloader = prepare_data(datamodule)

    trainer = cli.trainer
    trainer.fit(model, train_dataloader, val_dataloader)

    return trainer.callback_metrics['val_loss'].item()

In [ ]:
n_trials = 1
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=n_trials)